# Text-to-Image Evaluation

How do you measure if a generated image actually matches the text prompt?

This notebook covers **4 core metrics**:

| Metric | Question it answers | Needs reference images? |
|---|---|---|
| **CLIP Score** | Does the image match the prompt? | No |
| **FID** | Do generated images look like real images overall? | Yes |
| **Inception Score** | Are images high quality and diverse? | No |
| **Human Preference** | Which model's output do humans prefer? | No |

## Setup

In [1]:
import os, torch, numpy as np, requests
from PIL import Image
from io import BytesIO
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from groq import Groq
import json, re

groq_client = Groq(api_key=os.getenv('GROQ_API_KEY'))

def groq_chat(prompt, system='You are a helpful assistant.', temperature=0.0, max_tokens=600):
    resp = groq_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}],
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content.strip()

def parse_json(raw):
    raw = re.sub(r'```json|```', '', raw).strip()
    try: return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group()) if m else {}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cpu


## Download sample images

We simulate a T2I evaluation with 3 prompt-image pairs.
In practice, replace these with your model's actual outputs.

In [2]:
Path('t2i_eval/generated').mkdir(parents=True, exist_ok=True)
Path('t2i_eval/reference').mkdir(parents=True, exist_ok=True)

# Each pair: prompt + generated image + reference (ground truth) image
eval_pairs = [
    {'prompt': 'a golden retriever dog sitting on a grassy field',
     'gen_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/b/b3/01-Zwergschnauzer.jpg/320px-01-Zwergschnauzer.jpg',
     'ref_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg',
     'gen_file': 't2i_eval/generated/dog.jpg',
     'ref_file': 't2i_eval/reference/dog.jpg'},
    {'prompt': 'a red double-decker bus on a city street',
     'gen_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/36/A_London_bus_going_to_Elephant_%26_Castle.jpg/320px-A_London_bus_going_to_Elephant_%26_Castle.jpg',
     'ref_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/36/A_London_bus_going_to_Elephant_%26_Castle.jpg/320px-A_London_bus_going_to_Elephant_%26_Castle.jpg',
     'gen_file': 't2i_eval/generated/bus.jpg',
     'ref_file': 't2i_eval/reference/bus.jpg'},
    {'prompt': 'a snow-capped mountain with a clear blue sky',
     'gen_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg/320px-Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg',
     'ref_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg/320px-Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg',
     'gen_file': 't2i_eval/generated/mountain.jpg',
     'ref_file': 't2i_eval/reference/mountain.jpg'},
]

def download_image(url, path):
    if os.path.exists(path): return True
    try:
        r = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        Image.open(BytesIO(r.content)).convert('RGB').save(path)
        return True
    except Exception as e:
        print(f'  Download failed: {e}')
        return False

# Download or create fallback solid-color images
colors = [(200,150,100), (180,60,60), (100,140,200)]
for pair, color in zip(eval_pairs, colors):
    for key in ['gen', 'ref']:
        url, path = pair[f'{key}_url'], pair[f'{key}_file']
        if not download_image(url, path):
            Image.new('RGB', (224,224), color=color).save(path)
            print(f'  Created fallback: {path}')

print('Images ready!')

Images ready!


## 1. CLIP Score

**Idea:** CLIP was trained on 400M image-text pairs, so it knows if an image matches a description.

**How:** Encode the image and the text prompt into vectors, then measure cosine similarity.

- Score range: -1 to +1
- \> 0.25 = good match
- No reference image needed — just prompt + generated image

In [3]:
import open_clip
import torch.nn.functional as F

# Load CLIP model
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
clip_model.eval().to(device)

def clip_score(image_path, text):
    """Cosine similarity between image and text in CLIP space."""
    img = clip_preprocess(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    tok = clip_tokenizer([text]).to(device)
    with torch.no_grad():
        img_feat = F.normalize(clip_model.encode_image(img), dim=-1)
        txt_feat = F.normalize(clip_model.encode_text(tok), dim=-1)
    return (img_feat @ txt_feat.T).item()

# Evaluate
print('CLIP Score: Image-Text Alignment')
print('=' * 60)
for pair in eval_pairs:
    score = clip_score(pair['gen_file'], pair['prompt'])
    quality = 'Good' if score > 0.25 else 'Moderate' if score > 0.15 else 'Poor'
    print(f'  {pair["prompt"][:50]:<50}  {score:+.4f}  ({quality})')

/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP Score: Image-Text Alignment
  a golden retriever dog sitting on a grassy field    +0.2195  (Moderate)
  a red double-decker bus on a city street            +0.2151  (Moderate)
  a snow-capped mountain with a clear blue sky        +0.2351  (Moderate)


## 2. FID (Frechet Inception Distance)

**Idea:** Compare the *distribution* of generated images to real images. Not per-image — across the whole set.

**How:**
1. Extract features from all images using Inception-v3
2. Compute mean + covariance for generated set and reference set
3. Measure the distance between these two distributions

- 0 = identical distributions (perfect)
- < 50 = good, > 100 = poor
- Needs 10,000+ images for reliable results (we use 3 for demo)

In [4]:
from torchvision import models, transforms
from scipy import linalg

# Inception-v3 as feature extractor (remove classification head)
inception = models.inception_v3(weights='DEFAULT', transform_input=False)
inception.fc = torch.nn.Identity()  # output raw 2048-d features
inception.eval().to(device)

fid_preprocess = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

def get_features(image_paths):
    """Extract Inception features for a list of images."""
    feats = []
    with torch.no_grad():
        for p in image_paths:
            img = fid_preprocess(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
            feats.append(inception(img).squeeze().cpu().numpy())
    return np.array(feats)

def compute_fid(feats_real, feats_gen):
    """FID = distance between two Gaussian distributions fitted to the features."""
    mu_r, sig_r = feats_real.mean(0), np.cov(feats_real, rowvar=False)
    mu_g, sig_g = feats_gen.mean(0),  np.cov(feats_gen, rowvar=False)
    diff = mu_r - mu_g
    covmean, _ = linalg.sqrtm(sig_r @ sig_g, disp=False)
    if np.iscomplexobj(covmean): covmean = covmean.real
    return float(diff @ diff + np.trace(sig_r + sig_g - 2 * covmean))

# Compute
gen_paths = [p['gen_file'] for p in eval_pairs]
ref_paths = [p['ref_file'] for p in eval_pairs]

fid = compute_fid(get_features(ref_paths), get_features(gen_paths))
quality = 'Excellent' if fid < 10 else 'Good' if fid < 50 else 'Poor'

print('FID (Frechet Inception Distance)')
print('=' * 40)
print(f'  FID = {fid:.2f}  ({quality})')
print(f'  (Demo only — real FID needs 10k+ images)')

FID (Frechet Inception Distance)
  FID = -0.00  (Excellent)
  (Demo only — real FID needs 10k+ images)


## 3. Inception Score (IS)

**Idea:** Good T2I models produce images that are:
- **High quality** — Inception-v3 confidently recognizes what's in the image
- **Diverse** — different images trigger different classes

**How:** Feed images through Inception-v3, check if predictions are confident (quality) and varied (diversity).

- Real photos: IS ~ 11
- Good models: IS 7-10
- No reference images needed

In [5]:
from scipy.stats import entropy

# Full Inception-v3 with classification head (1000 ImageNet classes)
inception_is = models.inception_v3(weights='DEFAULT', transform_input=False)
inception_is.eval().to(device)

def inception_score(image_paths):
    """IS = exp(mean KL divergence between per-image predictions and overall average)."""
    preds = []
    with torch.no_grad():
        for p in image_paths:
            img = fid_preprocess(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
            logits = inception_is(img)
            if isinstance(logits, tuple): logits = logits[0]
            preds.append(F.softmax(logits, dim=1).squeeze().cpu().numpy())
    preds = np.array(preds)
    marginal = preds.mean(axis=0)                          # p(y) — average prediction
    kl_divs = [entropy(p, marginal) for p in preds]        # KL per image
    return float(np.exp(np.mean(kl_divs)))

is_score = inception_score(gen_paths)
quality = 'High' if is_score > 10 else 'Moderate' if is_score > 4 else 'Low'

print('Inception Score (IS)')
print('=' * 40)
print(f'  IS = {is_score:.4f}  ({quality})')

Inception Score (IS)
  IS = 1.0114  (Low)


## 4. Human Preference (LLM Judge)

**Idea:** Compare two models by describing their outputs in text, then ask an LLM which matches the prompt better.

**How:**
1. Describe each image in words (in practice, use a vision model like GPT-4V)
2. Send both descriptions + the original prompt to Groq
3. Groq picks the winner

This simulates human preference evaluation without actual human annotators.

In [6]:
JUDGE_SYS = '''You evaluate which image description better matches a text prompt.
Return JSON only: {"winner": "A" or "B", "score_a": 0.0-1.0, "score_b": 0.0-1.0, "reason": "..."}'''

# Manually provided descriptions (in practice, use a vision model to auto-describe)
descriptions = {
    'dog':      ('A small schnauzer with gray and white fur',
                 'A golden yellow labrador looking up with bright eyes'),
    'bus':      ('A red double-decker London bus on a city street',
                 'A red double-decker bus on a busy London road'),
    'mountain': ('A massive snow-covered mountain peak with clear blue sky',
                 'Mount Everest north face covered in snow with clear sky'),
}

print('Human Preference: LLM Judge')
print('=' * 60)

names = ['dog', 'bus', 'mountain']
for pair, name in zip(eval_pairs, names):
    desc_a, desc_b = descriptions[name]
    prompt = f'Prompt: {pair["prompt"]}\nImage A: {desc_a}\nImage B: {desc_b}\nWhich matches better?'
    result = parse_json(groq_chat(prompt, system=JUDGE_SYS))
    print(f'  Prompt: "{pair["prompt"][:50]}"')
    print(f'    A: {desc_a}')
    print(f'    B: {desc_b}')
    print(f'    Winner: {result.get("winner","?")} — {result.get("reason","")[:70]}')
    print()

Human Preference: LLM Judge
  Prompt: "a golden retriever dog sitting on a grassy field"
    A: A small schnauzer with gray and white fur
    B: A golden yellow labrador looking up with bright eyes
    Winner: B — Image B is a closer match because it features a dog with a similar coa

  Prompt: "a red double-decker bus on a city street"
    A: A red double-decker London bus on a city street
    B: A red double-decker bus on a busy London road
    Winner: A — Both images match the prompt, but Image A explicitly mentions 'city st

  Prompt: "a snow-capped mountain with a clear blue sky"
    A: A massive snow-covered mountain peak with clear blue sky
    B: Mount Everest north face covered in snow with clear sky
    Winner: A — Both images match the prompt, but Image A is more generic and directly

